# **Compulsory Assignment 3**
## *Machine Learning and Deep Learning (CDSCO2041C)*
*Group: MLS26_CA03 1*

*Student IDs: 185912, 161989, 160714 & 160363*

*Dataset: CIFAR-100 dataset*

---

## 1.1 Data Preparation

- Load CIFAR-100 train and test splits
- Create validation split (90/10) from training set
- Compute per-channel mean and standard deviation on training set only
- Normalize train, validation, and test images using those statistics
- Flatten normalized images to vectors of size 3072 (for Part 1.3)
- Keep spatial format (32x32x3) separately (for Part 1.2)

In [13]:
# Setup and imports

import os
os.environ['KERAS_HOME'] = os.path.join(os.path.dirname(os.path.abspath('pn_ml3.ipynb')), 'data')

import numpy as np
from tensorflow import keras
from tensorflow.keras import layers, Model
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import pickle

This was the code used to download the dataset to the local `data` folder:

```python
(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.cifar100.load_data()

print(f"Train images: {x_train_full.shape}")
print(f"Test images:  {x_test.shape}")
```

In [14]:
# Load CIFAR-100 from local data folder
data_dir = "data/datasets/cifar-100-python-target/cifar-100-python"

with open(f"{data_dir}/train", "rb") as f:
    train_batch = pickle.load(f, encoding="bytes")

with open(f"{data_dir}/test", "rb") as f:
    test_batch = pickle.load(f, encoding="bytes")

x_train_full = train_batch[b"data"].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
y_train_full = np.array(train_batch[b"fine_labels"])

x_test = test_batch[b"data"].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
y_test = np.array(test_batch[b"fine_labels"])

print(f"Train: {x_train_full.shape}  |  Test: {x_test.shape}")
print(f"Classes: {len(np.unique(y_train_full))}")

Train: (50000, 32, 32, 3)  |  Test: (10000, 32, 32, 3)
Classes: 100


After loading the now local dataset, we see that the dataset contains 50,000 training images and 10,000 test images, each labeled with one of 100 fine-grained classes. To create a validation set, we performed a stratified split of the training data, reserving 10% (5,000 images) for validation while keeping the class distribution proportional across all splits. This is crucial given the large number of classes, as a random split could lead to underrepresentation of some classes in smaller partitions.

In [15]:
# 90/10 stratified validation split
x_train, x_val, y_train, y_val = train_test_split(
    x_train_full, y_train_full,
    test_size=0.1,
    random_state=42,
    stratify=y_train_full
)

print(f"Train: {x_train.shape}  |  Val: {x_val.shape}")

Train: (45000, 32, 32, 3)  |  Val: (5000, 32, 32, 3)


In [16]:
# Per-channel mean and std computed on training set only
x_train_f = x_train.astype("float32")
x_val_f   = x_val.astype("float32")
x_test_f  = x_test.astype("float32")

channel_mean = x_train_f.mean(axis=(0, 1, 2))  # shape (3,)
channel_std  = x_train_f.std(axis=(0, 1, 2))   # shape (3,)

print(f"Channel means: {channel_mean}")
print(f"Channel stds:  {channel_std}")

# Normalize all splits using training statistics
x_train_norm = (x_train_f - channel_mean) / channel_std
x_val_norm   = (x_val_f   - channel_mean) / channel_std
x_test_norm  = (x_test_f  - channel_mean) / channel_std

# Flatten for Part 1.3 (pixel-based KNN classification)
x_train_flat = x_train_norm.reshape(x_train_norm.shape[0], -1)  # (45000, 3072)
x_val_flat   = x_val_norm.reshape(x_val_norm.shape[0], -1)      # (5000, 3072)
x_test_flat  = x_test_norm.reshape(x_test_norm.shape[0], -1)    # (10000, 3072)

print(f"\nFlat shapes:    train {x_train_flat.shape}, val {x_val_flat.shape}, test {x_test_flat.shape}")
print(f"Spatial shapes: train {x_train_norm.shape}  (kept for Parts 1.2 & 1.4)")

Channel means: [129.29881  124.03771  112.348564]
Channel stds:  [68.1466  65.36531 70.38802]

Flat shapes:    train (45000, 3072), val (5000, 3072), test (10000, 3072)
Spatial shapes: train (45000, 32, 32, 3)  (kept for Parts 1.2 & 1.4)


The CIFAR-100 dataset was loaded from local pickle files, yielding 50,000 training images and 10,000 test images across 100 fine-grained classes. A validation set was created by splitting 10% from the training data using stratified sampling, ensuring all 100 classes remain proportionally represented in both the 45,000-image training set and the 5,000-image validation set. Stratification is important here because a purely random split over 100 classes risks underrepresenting minority classes in smaller partitions.

Per-channel mean and standard deviation were computed on the training set only, then applied to normalize all three splits. Computing statistics solely on training data prevents information leakage from validation and test sets into the preprocessing step. Each normalized image was also flattened into a 3,072-dimensional vector for the pixel-based classification in Part 1.3, while the spatial 32x32x3 arrays were retained for the convolutional autoencoder (Part 1.2) and CNN (Part 1.4).

## 1.2 Autoencoder-Based Feature Learning
**Lecture:** L11 - Autoencoder, Attention & Transformers (autoencoder architecture, encoder-decoder, latent space, reconstruction loss)

Also review: L12 - CNN (Conv2D layers, pooling, convolutional blocks) since the autoencoder here is convolutional.

- Build convolutional autoencoder (encoder-decoder) with at least two conv blocks in the encoder and symmetric decoder
- Train on normalized spatial images (32x32x3) using MSE reconstruction loss
- Extract latent feature vectors from trained encoder for train, validation, and test sets

In [ ]:
# Build encoder


In [ ]:
# Build decoder


In [ ]:
# Train autoencoder


In [ ]:
# Extract latent features from encoder


## 1.3 Classification Using Pixel and Latent Features
**Lecture:** L04 - Supervised ML: KNN, Regression & Regularization (k-nearest neighbours, distance metrics, hyperparameters)

Also review: L06 - SVM & Performance Evaluation (accuracy, evaluation metrics, model comparison)

- Train KNN classifier on flattened pixel features (from 1.1)
- Train KNN classifier on latent features (from 1.2)
- Use same classifier type and hyperparameters for fair comparison
- Compare and report accuracy on validation/test sets

In [ ]:
# KNN on flattened pixel features


In [ ]:
# KNN on latent features


In [ ]:
# Compare results


## 1.4 CNN Model
**Lecture:** L12 - CNN (convolutional layers, feature maps, pooling, stride, padding, Conv2D architecture)

Also review: L08 - Gradient Descent, Backpropagation & Activation (loss functions, optimizers, training loop) and L09 - ANN, MLP, Dropout & Batch Norm (dense layers, dropout regularization, batch normalization)

- Build CNN with at least 3 Conv2D layers in 2+ convolutional blocks
- ReLU activations and MaxPooling after each block
- At least one dense layer before output
- Output layer: 100 classes, softmax activation
- Print number of trainable parameters

In [ ]:
# Build CNN architecture


In [ ]:
# Compile and train CNN


In [ ]:
# Print trainable parameters + evaluate
